# Lab 4 — หลักออกแบบชุดเดิม ด้วย LangChain middleware (EP06)

Lab 1–3 เราเขียนทุกอย่างเอง: tool ที่ชัด · retry · ตอบจาก cache · trace · วันที่ใน prompt
lab นี้ทำเรื่องเดียวกันด้วย `create_agent` ของ LangChain และ **middleware** — ชิ้นส่วนที่เสียบเข้าไปรอบๆ การเรียก model หรือ tool

| เราเขียนเองใน | ชิ้นส่วนของ LangChain |
|---|---|
| Lab 1: tool schema แบบ dict | `@tool` + docstring |
| Lab 1: `with_retry` | `ToolRetryMiddleware` |
| Lab 1: `run_degrade` + `final_answer` | `@wrap_tool_call` + `@after_agent` |
| Lab 2: `SYSTEM += วันที่` | `@dynamic_prompt` |
| Lab 2: `trace` + `show_trace` | `agent.stream(stream_mode="updates")` |
| Lab 3: `max_steps` / จำนวน call ที่ไม่แน่นอน | `ModelCallLimitMiddleware` |

คลัง paper และ search API ที่ล่มเป็น **ข้อมูลจำลองที่เขียนขึ้นเพื่อการสอน** ส่วนคำตอบของ model ทั้งหมดมาจากการรันจริง

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.3 — CC BY-SA 4.0

> ✏️ **Starter notebook — ฉบับให้ผู้เรียนลงมือเขียน**
>
> โค้ดส่วนใหญ่เตรียมไว้ให้แล้ว เขียนเพิ่มเฉพาะ **5 จุดที่มี `✏️ TODO`** (แทนที่ `___` หรือ `...` ด้วยโค้ดของคุณ) ทุกจุดบอกเวลาในวิดีโอที่อธิบายโค้ดตรงนั้น (▶ m:ss) — ดูคลิปก่อน แล้วลองเขียนเองโดยไม่เปิดเฉลย
>
> รันทุก cell ตามลำดับจากบนลงล่าง ถ้า cell ไหนยังเติมไม่ครบจะเจอ `NameError` หรือ `TypeError` เป็นเรื่องปกติ
>
> ติดตรงไหน เปิดเฉลยใน **`ep06-lab4-langchain-middleware.ipynb`** (โค้ดชุดเดียวกับในวิดีโอ)

## 0. เตรียมเครื่องก่อนเริ่ม

| สิ่งที่ต้องมี | รายละเอียด |
|---|---|
| Python 3.10 ขึ้นไป | หรือใช้ Google Colab ซึ่งมีให้แล้ว |
| OpenRouter API key | สมัครที่ openrouter.ai → Keys → Create Key แล้วเติมเครดิต (lab นี้ใช้ไม่ถึง 10 สตางค์) |
| Packages | `langchain`, `langgraph`, `langchain-openai`, `langchain-core`, `python-dotenv` (ล็อกเวอร์ชันชุดเดียวกับ EP05 Lab 5) |

**ใส่ API key**
- **Colab:** ไอคอน 🔑 Secrets → ชื่อ `OPENROUTER_API_KEY` → เปิด *Notebook access*
- **เครื่องตัวเอง:** สร้างไฟล์ `.env` ในโฟลเดอร์เดียวกับ notebook แล้วใส่บรรทัด `OPENROUTER_API_KEY=sk-or-...`

วิธีตั้งค่า key และเหตุผลที่เลือก model อยู่ใน Lab 1

In [ ]:
%pip install -q "langchain==1.4.7" "langgraph==1.2.16" "langchain-openai==1.7.2" "langchain-core==1.7.0" python-dotenv

## 1. model + คลัง paper + search API ที่ล่มได้

`ChatOpenAI` ชี้ไป OpenRouter (Gemma 4 26B ล็อกที่ DeepInfra) · คลัง `PAPERS`, `find_papers`, `SearchTimeout` และ `down` ยกมาจาก Lab 1–2

In [ ]:
import json, os, datetime
from langchain_openai import ChatOpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

In [ ]:
llm = ChatOpenAI(model="google/gemma-4-26b-a4b-it", base_url="https://openrouter.ai/api/v1", api_key=load_key(),
                 extra_body={"reasoning": {"enabled": False},
                             "provider": {"order": ["deepinfra/fp8"], "allow_fallbacks": False}})

In [ ]:
PAPERS = [
    {"id": "P01", "title": "Informer: efficient transformer for long sequence time-series forecasting", "year": 2021},
    {"id": "P02", "title": "Autoformer: decomposition transformers for long-term series forecasting", "year": 2021},
    {"id": "P03", "title": "FEDformer: frequency enhanced decomposed transformer for time series", "year": 2022},
    {"id": "P04", "title": "Are transformers effective for time series forecasting?", "year": 2023},
    {"id": "P05", "title": "PatchTST: a time series is worth 64 words", "year": 2023},
    {"id": "P06", "title": "iTransformer: inverted transformers for time series forecasting", "year": 2024},
    {"id": "P07", "title": "TimesFM: a decoder-only foundation model for time-series forecasting", "year": 2024},
    {"id": "P08", "title": "Chronos: learning the language of time series", "year": 2024},
    {"id": "P09", "title": "Attention is all you need", "year": 2017},
    {"id": "P10", "title": "BERT: pre-training of deep bidirectional transformers", "year": 2019},
    {"id": "P11", "title": "Temporal fusion transformers for interpretable time series forecasting", "year": 2019},
    {"id": "P12", "title": "Moirai: unified training of universal time series forecasting transformers", "year": 2024},
]
STOP = {"with", "from", "paper", "papers", "model", "models", "using", "published", "since", "research", "about", "learning"}

def find_papers(query, year_from=0, library=PAPERS):
    words = [w for w in query.lower().replace("-", " ").split() if len(w) > 3 and w not in STOP]
    return [p for p in library
            if p["year"] >= year_from and any(w in p["title"].lower().replace("-", " ") for w in words)]

class SearchTimeout(Exception):
    pass

down = {"fails_left": 0}

## 2. ③ tool ที่ดี: `@tool` + docstring

ชื่อฟังก์ชัน → ชื่อ tool · type hint → ชนิดของ argument · docstring → คำอธิบาย (`parse_docstring=True` อ่านส่วน `Args:` เป็นคำอธิบายของแต่ละ argument)

In [ ]:
# ✏️ TODO 1 (▶ วิดีโอ 1:46): เขียน docstring ส่วน Args ให้ model รู้ว่า query และ year_from คืออะไร
from langchain_core.tools import tool

@tool(parse_docstring=True)
def search_papers(query: str, year_from: int) -> list[dict]:
    """ค้น paper วิชาการในคลังของ Cogito Learn ตามหัวข้อ คืน list ของ {id, title, year} ถ้าไม่เจอคืน []

    Args:
        query: ___
        year_from: ___
    """
    if down["fails_left"] > 0:
        down["fails_left"] -= 1
        print("   ⚠️ search API timeout")
        raise SearchTimeout("search API ไม่ตอบภายใน 10 วินาที")
    return find_papers(query, year_from)

print(json.dumps(search_papers.args, ensure_ascii=False, indent=1))

## 3. ② trace: `stream_mode="updates"`

`create_agent` สร้าง graph ของ LangGraph ให้ · `stream` ส่งผลของทุก node ออกมาทีละขั้น — node ของ model, tool และของ middleware บางตัว

In [ ]:
def ask(agent, question):
    for update in agent.stream({"messages": [{"role": "user", "content": question}]}, stream_mode="updates"):
        for node, value in update.items():
            for m in (value or {}).get("messages", []):
                calls = getattr(m, "tool_calls", None)
                if calls:
                    what = f"🔧 ขอ {[(c['name'], c['args']) for c in calls]}"
                elif m.type == "tool":
                    what = f"📄 ผล: {str(m.content)[:90]}"
                else:
                    what = f"💬 ตอบ: {str(m.content)[:90]!r}"
                print(f"[{node}] {what}")
    return m.content

## 4. `create_agent` + `@dynamic_prompt`

In [ ]:
from langchain.agents import create_agent

SYSTEM = "คุณคือผู้ช่วยค้น paper ของ Cogito Learn ใช้ tool หาข้อมูลจริงเสมอ ห้ามตอบจากความจำ"
RECENT = "หา paper เรื่อง transformer สำหรับ time-series ช่วง 2 ปีล่าสุด"

plain = create_agent(llm, tools=[search_papers], system_prompt=SYSTEM)
answer = ask(plain, RECENT)

In [ ]:
# ✏️ TODO 2 (▶ วิดีโอ 3:37): คืน system prompt ที่ต่อท้ายด้วยวันที่ของวันนี้
from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def with_today(request):
    return ___

dated = create_agent(llm, tools=[search_papers], middleware=[with_today])
answer = ask(dated, RECENT)

## 5. ④ เผื่อระบบล่ม: retry → cache → แปะคำเตือน

middleware ที่ห่อการเรียก tool ทำงานซ้อนกันตามลำดับใน list — **ตัวแรกอยู่นอกสุด**:
`cache_fallback` ( `ToolRetryMiddleware` ( tool ) ) → retry ก่อน ถ้ายังล่มค่อยตกไปที่ cache

In [ ]:
# ✏️ TODO 3 (▶ วิดีโอ 4:23): ตั้ง ToolRetryMiddleware ให้ retry เฉพาะ SearchTimeout
from langchain.agents.middleware import ToolRetryMiddleware

retry = ToolRetryMiddleware(retry_on=___, max_retries=2,
                            initial_delay=1, backoff_factor=2, jitter=False, on_failure="error")

In [ ]:
from langchain.agents.middleware import wrap_tool_call, after_agent
from langchain_core.messages import ToolMessage, AIMessage

SNAPSHOT_DATE = "2024-01-15"
SNAPSHOT = [p for p in PAPERS if p["year"] <= 2023]
notes = []

@wrap_tool_call
def cache_fallback(request, handler):
    try:
        return handler(request)
    # ✏️ TODO 4 (▶ วิดีโอ 5:07): จับ SearchTimeout ที่หลุดมาจาก retry แล้วค้นจาก SNAPSHOT แทน
    except ___:
        args = request.tool_call["args"]
        notes.append(f"⚠️ search ใช้ไม่ได้ ผลนี้มาจากสำเนาคลังวันที่ {SNAPSHOT_DATE} อาจไม่มี paper ใหม่")
        papers = ___
        return ToolMessage(json.dumps({"source": f"สำเนาคลังวันที่ {SNAPSHOT_DATE} — อาจไม่ใหม่", "papers": papers},
                                      ensure_ascii=False), tool_call_id=request.tool_call["id"])

@after_agent
def add_notes(state, runtime):
    if notes:
        last = state["messages"][-1]
        return {"messages": [AIMessage("\n".join(notes) + "\n" + last.content, id=last.id)]}

In [ ]:
# ✏️ TODO 5 (▶ วิดีโอ 5:42): เรียง middleware ให้ cache_fallback อยู่ชั้นนอกของ retry
from langchain.agents.middleware import ModelCallLimitMiddleware

agent = create_agent(llm, tools=[search_papers],
                     middleware=[with_today, ___, ___, add_notes,
                                 ModelCallLimitMiddleware(run_limit=6)])

In [ ]:
Q2023 = "หา paper เรื่อง transformer สำหรับ time-series ที่ออกตั้งแต่ปี 2023 ขึ้นไป"

down["fails_left"] = 2
answer = ask(agent, Q2023)

In [ ]:
notes.clear()
down["fails_left"] = 99
answer = ask(agent, Q2023)
print(answer)

## สรุป

| หลัก | เราเขียนเอง (Lab 1–3) | LangChain |
|---|---|---|
| ③ tool ที่ดี | schema แบบ dict | `@tool(parse_docstring=True)` |
| ② trace | `trace` + `show_trace` | `stream_mode="updates"` — middleware ที่เพิ่ม message ก็โผล่เป็น node ของตัวเอง (`add_notes.after_agent`) |
| ข้อมูลที่ model ต้องรู้ | `SYSTEM += วันที่` | `@dynamic_prompt` |
| ④ retry | `with_retry` | `ToolRetryMiddleware(retry_on=…, on_failure="error")` |
| ④ graceful degradation | `run_degrade` + `final_answer` | `@wrap_tool_call` (ชั้นนอกของ retry) + `@after_agent` |
| จำนวน call ไม่แน่นอน | `max_steps` | `ModelCallLimitMiddleware(run_limit=6)` |

framework ทำให้โค้ดสั้นลง แต่ **ไม่ได้ตัดสินใจแทนเรา**: ต้องรู้เองว่า retry error ชนิดไหน, middleware เรียงอย่างไร, และอะไรที่ผู้ใช้ต้องเห็น

**แบบฝึกหัด**
1. ตั้ง `ModelCallLimitMiddleware(run_limit=1)` แล้วถามคำถามเดิม — agent หยุดตรงไหน และตอบว่าอะไร?
2. สลับลำดับ `cache_fallback` กับ `retry` ใน list แล้วให้ search ล่ม 2 ครั้ง — ผลต่างจากเดิมอย่างไร? ดูจาก trace

**ถัดไป — สรุป EP06:** ทบทวนหลักออกแบบหกข้อ และวิธีเลือก pattern จากสิ่งที่เราเห็นในสี่แล็บ